## CRF - baseline

In [99]:
%pip install -r ../requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [100]:
# fix on the imports
import sys
from pathlib import Path

sys.path.append(str(Path().resolve().parent))

In [112]:
# parameters

NO_SAMPLES : int   = 50000
SEED:      int   = 42
N_FOLDS:   int   = 3

CV = 3
N_ITER = 20
N_JOBS = -1

In [102]:

# preprocessing data with commmon project method
from data_preparation.preprocess_data import preprocess_data

df = preprocess_data()[:NO_SAMPLES]

Loading dataset...
Cleaning data...


In [103]:
print(df['target'][0])
print(df['input'][0])

{'house_number': '81', 'street': 'Epovska', 'city': 'Lubinjë E Epërme', 'country': 'XK', 'postal_code': '', 'state': ''}
Ep Epovska 81 Lubihjë XK Epërme Epërme


In [104]:
from rapidfuzz import fuzz

def align_labels(raw_address: str, parsed: dict) -> tuple:
    tokens = raw_address.split()
    labels = ["O"] * len(tokens)

    VALID_FIELDS = {"house_number", "street", "city", "postal_code", "state", "country"}

    MAX_TOKENS = {
        "city":   3,
        "street": 4,
        "state":  3,
    }

    sorted_fields = sorted(
        [f for f in parsed if f in VALID_FIELDS and parsed[f]],
        key=lambda f: len(str(parsed[f])),
        reverse=True
    )

    for field in sorted_fields:
        value_tokens = str(parsed[field]).split()
        window_size = min(len(value_tokens), MAX_TOKENS.get(field, len(value_tokens)))

        best_score, best_i = 0, -1

        for i in range(len(tokens) - window_size + 1):
            if any(labels[i + k] != "O" for k in range(window_size)):
                continue

            window    = " ".join(tokens[i:i + window_size])
            candidate = " ".join(value_tokens[:window_size])

            # trying to find exact match of the label
            if window.lower() == candidate.lower():
                best_score, best_i = 100, i
                break

            # fuzzy matching if there is no exact label for a token
            score = fuzz.ratio(window.lower(), candidate.lower())
            if score > best_score:
                best_score, best_i = score, i

        if best_score >= 60 and best_i >= 0:
            labels[best_i] = f"B-{field}"
            for k in range(1, window_size):
                labels[best_i + k] = f"I-{field}"

    return tuple(tokens), tuple(labels)

In [105]:
test_adddress = df['input'][0]
test_parsed = df['target'][0]

tokens, labels = align_labels(test_adddress, test_parsed)
print("Tokens:", tokens)
print("Labels:", labels)

Tokens: ('Ep', 'Epovska', '81', 'Lubihjë', 'XK', 'Epërme', 'Epërme')
Labels: ('O', 'B-street', 'B-house_number', 'B-city', 'I-city', 'I-city', 'O')


In [106]:
import regex as re

def word_features(tokens: list[str], i: int) -> dict:
    word = tokens[i]
    features = {
        "bias":           1.0,
        "word.lower":     word.lower(),
        "word.isupper":   word.isupper(),
        "word.istitle":   word.istitle(),
        "word.isdigit":   word.isdigit(),
        "word.len":       len(word),
        "prefix2":        word[:2].lower(),
        "suffix2":        word[-2:].lower(),
        "is_num_pattern": bool(re.match(r'^\d+[a-zA-Z]?$', word)),
        "is_postal":      bool(re.match(r'\b[A-Za-z0-9][A-Za-z0-9\- ]{2,9}[A-Za-z0-9]\b', word)),
        "is_cc":          bool(re.match(r'^[A-Z]{2}$', word)),
        "is_first":         i == 0,
        "is_last":          i == len(tokens) - 1,
        "position":         i,
        "position_rel":     round(i / max(len(tokens) - 1, 1), 2),
    }

    # Context window ±2
    for offset, prefix in [(-2,"m2"), (-1,"m1"), (1,"p1"), (2,"p2")]:
        j = i + offset
        if 0 <= j < len(tokens):
            features[f"{prefix}:word.lower"] = tokens[j].lower()
            features[f"{prefix}:isdigit"]    = tokens[j].isdigit()
            features[f"{prefix}:istitle"]    = tokens[j].istitle()
        else:
            features[f"{prefix}:BOS_EOS"] = True

    return features

In [110]:
test_tokens = ['Ep', 'Epovska', '81', 'Lubihjë', 'XK', 'Epërme', 'Epërme']
for i in range(len(test_tokens)):
    print(f"Features for '{test_tokens[i]}':")
    print(word_features(test_tokens, i))

Features for 'Ep':
{'bias': 1.0, 'word.lower': 'ep', 'word.isupper': False, 'word.istitle': True, 'word.isdigit': False, 'word.len': 2, 'prefix2': 'ep', 'suffix2': 'ep', 'is_num_pattern': False, 'is_postal': False, 'is_cc': False, 'is_first': True, 'is_last': False, 'position': 0, 'position_rel': 0.0, 'm2:BOS_EOS': True, 'm1:BOS_EOS': True, 'p1:word.lower': 'epovska', 'p1:isdigit': False, 'p1:istitle': True, 'p2:word.lower': '81', 'p2:isdigit': True, 'p2:istitle': False}
Features for 'Epovska':
{'bias': 1.0, 'word.lower': 'epovska', 'word.isupper': False, 'word.istitle': True, 'word.isdigit': False, 'word.len': 7, 'prefix2': 'ep', 'suffix2': 'ka', 'is_num_pattern': False, 'is_postal': True, 'is_cc': False, 'is_first': False, 'is_last': False, 'position': 1, 'position_rel': 0.17, 'm2:BOS_EOS': True, 'm1:word.lower': 'ep', 'm1:isdigit': False, 'm1:istitle': True, 'p1:word.lower': '81', 'p1:isdigit': True, 'p1:istitle': False, 'p2:word.lower': 'lubihjë', 'p2:isdigit': False, 'p2:istitle':

In [111]:
import pandas as pd

def prepare_split(split_df: pd.DataFrame):
    tokenized_addresses = []
    aligned_labels = []

    for _, row in split_df.iterrows():
        if row['target'].get('country') == 'Taiwan':
            continue

        raw_address = row['input']
        target_dict = row['target']
        tokens, labels = align_labels(raw_address, target_dict)

        if not tokens:
            continue

        tokenized_addresses.append(tokens)
        aligned_labels.append(labels)

    return tokenized_addresses, aligned_labels

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
import scipy.stats as stats
from data_preparation.preprocess_data import make_splits
import regex as re
import sklearn_crfsuite

# Looking for optimal parameters on fold 0 only
print("Tuning hyperparameters on fold 0...")

first_split = next(make_splits(df))
train_tokens, train_labels = prepare_split(first_split['train'])
val_tokens,   val_labels   = prepare_split(first_split['val'])

X_train = [[word_features(list(t), i) for i in range(len(t))] for t in train_tokens]
X_val   = [[word_features(list(t), i) for i in range(len(t))] for t in val_tokens]
y_train = [list(l) for l in train_labels]
y_val   = [list(l) for l in val_labels]

X_tune = X_train + X_val
y_tune = y_train + y_val

crf = sklearn_crfsuite.CRF(
    algorithm="lbfgs",
    max_iterations=200,
    all_possible_transitions=True,
)

param_distributions = {
    "c1": stats.expon(scale=0.5),  
    "c2": stats.expon(scale=0.05),
}

search = RandomizedSearchCV(
    crf,
    param_distributions,
    cv=CV,
    n_iter=N_ITER,
    scoring="f1_weighted", 
    n_jobs=N_JOBS,
    verbose=1,
    random_state=SEED,
)
search.fit(X_tune, y_tune)

best_c1 = search.best_params_["c1"]
best_c2 = search.best_params_["c2"]
print(f"Best params → c1: {best_c1:.4f} | c2: {best_c2:.4f}")

Tuning hyperparameters on fold 0...
Downsampling data from 50000 to 10000 samples...
Fitting 3 folds for each of 20 candidates, totalling 60 fits


/Users/weronikadomczewska/Documents/magisterskie/semestr3/NLP/NLP-address-data-parsing/.venv/lib/python3.13/site-packages/sklearn/model_selection/_validation.py:927: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "/Users/weronikadomczewska/Documents/magisterskie/semestr3/NLP/NLP-address-data-parsing/.venv/lib/python3.13/site-packages/sklearn/model_selection/_validation.py", line 916, in _score
    scores = scorer(estimator, X_test, y_test, **score_params)
  File "/Users/weronikadomczewska/Documents/magisterskie/semestr3/NLP/NLP-address-data-parsing/.venv/lib/python3.13/site-packages/sklearn/metrics/_scorer.py", line 317, in __call__
    return self._score(partial(_cached_call, None), estimator, X, y_true, **_kwargs)
           ~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/weronikadomczewska/Documents/magisterskie/semestr3/NLP/NLP-a

Best params → c1: 0.2346 | c2: 0.1505


# Libpostal - just for test

In [114]:
%pip install postal

Note: you may need to restart the kernel to use updated packages.


In [116]:
from postal.parser import parse_address


def libpostal_predict(raw_address: str, tokens: tuple) -> list[str]:
    parsed = parse_address(raw_address)

    value_to_field = {}
    for value, component in parsed:
        field = (component
                 .replace("road",     "street")
                 .replace("postcode", "postal_code")
                 .replace("house",    "house_number"))

        if field not in {"house_number", "street", "city", "postal_code", "state", "country"}:
            continue

        for sub_token in value.lower().split():
            value_to_field[sub_token] = field

    labels = ["O"] * len(tokens)
    prev_field = None

    for i, token in enumerate(tokens):
        field = value_to_field.get(token.lower())
        if field:
            labels[i] = f"I-{field}" if field == prev_field else f"B-{field}"
            prev_field = field
        else:
            prev_field = None

    return labels


# ── Ewaluacja na zbiorze testowym ─────────────────────────────────────────────
from sklearn_crfsuite import metrics
from sklearn.metrics import classification_report

def evaluate_libpostal(test_df: pd.DataFrame):
    y_true = []
    y_pred = []

    for _, row in test_df.iterrows():
        if row['target'].get('country') == 'Taiwan':
            continue

        raw_address = row['input']
        target_dict = row['target']

        tokens, true_labels = align_labels(raw_address, target_dict)

        if not tokens:
            continue

        pred_labels = libpostal_predict(raw_address, tokens)

        y_true.append(list(true_labels))
        y_pred.append(pred_labels)

    label_names = list({l for seq in y_true for l in seq if l != "O"})
    print(metrics.flat_classification_report(y_true, y_pred, labels=sorted(label_names)))

    return y_true, y_pred

In [117]:
def bio_to_dict(tokens: tuple, labels: list[str]) -> dict:
    result = {}
    current_field, current_tokens = None, []

    for token, label in zip(tokens, labels):
        if label.startswith("B-"):
            if current_field:
                result[current_field] = " ".join(current_tokens)
            current_field = label[2:]
            current_tokens = [token]
        elif label.startswith("I-") and label[2:] == current_field:
            current_tokens.append(token)
        else:
            if current_field:
                result[current_field] = " ".join(current_tokens)
            current_field, current_tokens = None, []

    if current_field:
        result[current_field] = " ".join(current_tokens)

    return result

In [118]:
from data_preparation.preprocess_data import evaluate_predictions

for fold_num, split in enumerate(make_splits(df)):
    print(f"\n═══════════ Fold {fold_num} ═══════════")

    train_tokens, train_labels = prepare_split(split['train'])
    test_tokens,  test_labels  = prepare_split(split['test'])

    X_train = [[word_features(list(t), i) for i in range(len(t))] for t in train_tokens]
    X_test  = [[word_features(list(t), i) for i in range(len(t))] for t in test_tokens]
    y_train = [list(l) for l in train_labels]
    y_test  = [list(l) for l in test_labels]

    crf = sklearn_crfsuite.CRF(
        algorithm="lbfgs", c1=best_c1, c2=best_c2,
        max_iterations=200, all_possible_transitions=True,
    )
    crf.fit(X_train, y_train)
    y_crf_pred = crf.predict(X_test)

    label_names = sorted([l for l in crf.classes_ if l != "O"])

    # ── Token-level (BIO) ────────────────────────────────────────────────
    print("\n── CRF token-level ──")
    print(metrics.flat_classification_report(y_test, y_crf_pred, labels=label_names))

    # ── Field-level ──────────────────────────────────────────────────────
    gold_dicts     = list(split['test']['target'])
    crf_pred_dicts = [bio_to_dict(t, l) for t, l in zip(test_tokens, y_crf_pred)]
    crf_results    = evaluate_predictions(gold_dicts, crf_pred_dicts)

    print("\n── CRF field-level ──")
    print(f"Exact match:       {crf_results['exact_match']:.3f}")
    print(f"Overall item acc:  {crf_results['overall_item_accuracy']:.3f}")
    for field, acc in crf_results['per_field_accuracy'].items():
        print(f"  {field:<15} {acc:.3f}")

    # ── libpostal ────────────────────────────────────────────────────────
    print("\n── libpostal ──")
    evaluate_libpostal(split['test'])

Downsampling data from 50000 to 10000 samples...

═══════════ Fold 0 ═══════════

── CRF token-level ──
                precision    recall  f1-score   support

        B-city       0.82      0.86      0.84      1456
     B-country       0.97      0.98      0.97      1286
B-house_number       0.94      0.95      0.95      1435
 B-postal_code       0.89      0.93      0.91       219
       B-state       0.97      0.97      0.97      1284
      B-street       0.80      0.81      0.81      1410
        I-city       0.46      0.44      0.45        68
       I-state       0.82      0.78      0.80        23
      I-street       0.69      0.70      0.69       819

     micro avg       0.87      0.89      0.88      8000
     macro avg       0.82      0.82      0.82      8000
  weighted avg       0.87      0.89      0.88      8000


── CRF field-level ──
Exact match:       0.299
Overall item acc:  0.789
  house_number    0.853
  street          0.521
  city            0.693
  country         0.